In [ ]:
!pip install seisbench
import seisbench
import seisbench.data as sbd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import obspy

    obspy.read()
except TypeError:
    # Needs to restart the runtime once, because obspy only works properly after restart.
    print(
        "Stopping RUNTIME. If you run this code for the first time, this is expected. Colaboratory will restart automatically. Please run again."
    )
    exit()

In [ ]:
# Carica il parquet
df_signals = pd.read_parquet('/kaggle/input/dataset/acc_preprocessed_scaling.parquet')

# Per ogni stazione, estrai le 3 componenti
for station_code in df_signals['file'].apply(get_station_from_filename).unique():
    # Filtra i 3 file di questa stazione
    mask = df_signals['file'].apply(lambda f: get_station_from_filename(f) == station_code)
    df_station = df_signals[mask].copy()
    
    # Pivot per ottenere le 3 componenti allineate
    # (assumendo che tutte e 3 abbiano stesso numero di sample)
    components = {}
    for file_name in df_station['file'].unique():
        component = get_component_from_filename(file_name)
        df_comp = df_station[df_station['file'] == file_name].sort_values('sample')
        components[component] = df_comp['acceleration'].values
    
    # Stack in array [n_samples, 3]
    waveform = np.column_stack([components['HNZ'], components['HNN'], components['HNE']])


In [ ]:
pn_model = sbm.PhaseNet.from_pretrained("ethz")
eqt_model = sbm.EQTransformer.from_pretrained("ethz")
gpd_model = sbm.GPD.from_pretrained("ethz")